# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Install mlcroissant if not yet installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields. All entity references use their `@id`.

In [ ]:
record_sets = list(dataset.record_sets)

print("Available record sets:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name']}")
    print(f"  Description: {rs.get('description', '')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    if fields:
        print("  Fields:")
        for fld in fields:
            if isinstance(fld, dict):
                field_id = fld.get('@id')
                field_name = fld.get('name', '')
            else:
                field_id = fld
                field_name = ''
            print(f"    - {field_id} {field_name}")
    else:
        print("  No fields listed.")

## 3. Data Extraction
Load data from a record set into a DataFrame for analysis.

Replace `<record_set_id>` and field IDs by their `@id` based on the above table.


In [ ]:
# List the @id values for each record set for reference
record_set_ids = [rs['@id'] for rs in record_sets]
print("Record set @id's:", record_set_ids)

# We'll use the first record set for demonstration (update this if needed)
selected_record_set_id = record_set_ids[0]

# Extract records for the selected record set
records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)

# Print the columns and a preview of the data
print(f"Columns in {selected_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply processing such as filtering or normalizing numeric fields.

We first examine available numeric fields:


In [ ]:
# Infer which fields are numeric. For demonstration, pick an obvious numeric field, such as the GAD-7 score.
# Usually fields like 'gad7_score', 'phq9_score', or 'psq_score' are present. List columns for confirmation.
print(df.columns.tolist())
# Example: assume 'gad7_score' is an integer field in the data (replace this with the true @id from the schema if available!)
numeric_field = 'gad7_score'  # Replace with the true field @id if different.
if numeric_field not in df.columns:
    raise ValueError("Please update 'numeric_field' with an available numeric column from the dataset.")

threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a demographic field (e.g., 'gender', replace with the @id as appropriate)
group_field = 'gender'  # Replace with the true field @id if different
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field, dropna=True).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric assessment score (e.g., GAD-7).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field} scores")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If there's a demographic field, show boxplot by gender
if group_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded metadata and records from a Croissant-described dataset using `mlcroissant`.
- Explored available record sets and their fields via `@id`.
- Loaded survey responses into Pandas DataFrames for analysis.
- Performed initial EDA to filter, normalize, and visualize response data.

This workflow can be adapted to any Croissant-standard dataset, supporting AI-ready, reproducible, and FAIR analysis pipelines.